In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
corpus =  ["Python is a popular programming language", "Machine learning helps computers learn from data", "Python is used for data analysis and software development", "Healthcare organizations use artificial intelligence to detect diseases", "Banks use machine learning to identify fraudulent transactions", "Data science helps businesses analyze information and make better decisions"]


In [8]:
v = TfidfVectorizer()
v.fit(corpus)
transform_output =  v.transform(corpus)


In [10]:
print(v.vocabulary_)

{'python': 30, 'is': 21, 'popular': 28, 'programming': 29, 'language': 22, 'machine': 25, 'learning': 24, 'helps': 17, 'computers': 7, 'learn': 23, 'from': 15, 'data': 8, 'used': 36, 'for': 13, 'analysis': 0, 'and': 2, 'software': 32, 'development': 11, 'healthcare': 16, 'organizations': 27, 'use': 35, 'artificial': 3, 'intelligence': 20, 'to': 33, 'detect': 10, 'diseases': 12, 'banks': 4, 'identify': 18, 'fraudulent': 14, 'transactions': 34, 'science': 31, 'businesses': 6, 'analyze': 1, 'information': 19, 'make': 26, 'better': 5, 'decisions': 9}


In [14]:
for word in zip (v.get_feature_names_out(),v.idf_):
    print(word)  #IDF value of each word

('analysis', 2.252762968495368)
('analyze', 2.252762968495368)
('and', 1.8472978603872037)
('artificial', 2.252762968495368)
('banks', 2.252762968495368)
('better', 2.252762968495368)
('businesses', 2.252762968495368)
('computers', 2.252762968495368)
('data', 1.5596157879354227)
('decisions', 2.252762968495368)
('detect', 2.252762968495368)
('development', 2.252762968495368)
('diseases', 2.252762968495368)
('for', 2.252762968495368)
('fraudulent', 2.252762968495368)
('from', 2.252762968495368)
('healthcare', 2.252762968495368)
('helps', 1.8472978603872037)
('identify', 2.252762968495368)
('information', 2.252762968495368)
('intelligence', 2.252762968495368)
('is', 1.8472978603872037)
('language', 2.252762968495368)
('learn', 2.252762968495368)
('learning', 1.8472978603872037)
('machine', 1.8472978603872037)
('make', 2.252762968495368)
('organizations', 2.252762968495368)
('popular', 2.252762968495368)
('programming', 2.252762968495368)
('python', 1.8472978603872037)
('science', 2.25276

In [17]:
print(transform_output.toarray())

[[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.39339985 0.47974754 0.
  0.         0.         0.         0.         0.47974754 0.47974754
  0.39339985 0.         0.         0.         0.         0.
  0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.42653458 0.29529519 0.         0.         0.
  0.         0.         0.         0.42653458 0.         0.34976445
  0.         0.         0.         0.         0.         0.42653458
  0.34976445 0.34976445 0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.        ]
 [0.36523197 0.         0.29949544 0.         0.         0.
  0.         0.         0.25285463 0.         0.         0.36523197
  0.         0.36523197 0.         0.         0.         0.
  0.         0.         0.         0.299

#### Problem Statement: Given a description about a product sold on e-commerce website, classify it in one of the 4 categories

In [27]:
import pandas as pd
df =pd.read_csv("Ecommerce_data.csv")
df.head()


,Text,label
0,Urban Ladder Eisner Low Back Study-Office Comp...,Household
1,"Contrast living Wooden Decorative Box,Painted ...",Household
2,IO Crest SY-PCI40010 PCI RAID Host Controller ...,Electronics
3,ISAKAA Baby Socks from Just Born to 8 Years- P...,Clothing & Accessories
4,Indira Designer Women's Art Mysore Silk Saree ...,Clothing & Accessories


In [29]:
df.label.value_counts()

label
Household                 6000
Electronics               6000
Clothing & Accessories    6000
Books                     6000
Name: count, dtype: int64

 There is no problem of class imbalance and hence no need to apply any balancing techniques like undersampling, oversampling etc.

In [38]:
df['label_num'] = df.label.map({'Household' : 0, 
    'Books': 1, 
    'Electronics': 2, 
    'Clothing & Accessories': 3
})
df.head(5)

,Text,label,label_num
0,Urban Ladder Eisner Low Back Study-Office Comp...,Household,0
1,"Contrast living Wooden Decorative Box,Painted ...",Household,0
2,IO Crest SY-PCI40010 PCI RAID Host Controller ...,Electronics,2
3,ISAKAA Baby Socks from Just Born to 8 Years- P...,Clothing & Accessories,3
4,Indira Designer Women's Art Mysore Silk Saree ...,Clothing & Accessories,3


Training Model without the preprocessing

In [43]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test=train_test_split(
    df.Text, 
    df.label_num, 
    test_size=0.2, # 20% samples will go to test dataset
    random_state=2022, #Every time you run the code with the same dataset and settings, you get the same split.
    stratify=df.label_num
)


In [49]:
print(X_train.shape)
print(X_test.shape)

(19200,)
(4800,)


In [53]:
y_train.value_counts(),y_test.value_counts()

(label
 0    4800
 2    4800
 3    4800
 1    4800
 Name: count, dtype: int64,
 label
 0    1200
 2    1200
 3    1200
 1    1200
 Name: count, dtype: int64)

**Attempt 1** :

1. using sklearn pipeline module create a classification pipeline to classify the Ecommerce Data.

In [62]:
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

clf = Pipeline([
    ('vectorizer_tfidf', TfidfVectorizer()),
    ('KNN', KNeighborsClassifier())
])

clf.fit(X_train,y_train)
y_pred =clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95      1200
           1       0.97      0.95      0.96      1200
           2       0.97      0.97      0.97      1200
           3       0.97      0.98      0.97      1200

    accuracy                           0.96      4800
   macro avg       0.96      0.96      0.96      4800
weighted avg       0.96      0.96      0.96      4800



**Attempt 2** :

1. using sklearn pipeline module create a classification pipeline to classify the Ecommerce Data.

**Note:**
- use TF-IDF for pre-processing the text.

- use **MultinomialNB** as the classifier.

In [69]:
from sklearn.naive_bayes import MultinomialNB

clf = Pipeline([
     ('vectorizer_tfidf',TfidfVectorizer()),    
     ('Multi NB', MultinomialNB())         
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.96      0.94      1200
           1       0.98      0.92      0.95      1200
           2       0.97      0.97      0.97      1200
           3       0.97      0.99      0.98      1200

    accuracy                           0.96      4800
   macro avg       0.96      0.96      0.96      4800
weighted avg       0.96      0.96      0.96      4800



#### Use text pre-processing to remove stop words, punctuations and apply lemmatization

In [74]:
import spacy
nlp = spacy.load("en_core_web_sm")
def preprocess(text):
    doc = nlp(text)
    filtered_tokens = []
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        filtered_tokens.append(token.lemma_)
    return " ".join(filtered_tokens)   #It converts the filtered list of words back into a string so that TfidfVectorizer() can process the text.

In [76]:
df['preprocessed_txt'] = df['Text'].apply(preprocess) 
df.head()

,Text,label,label_num,preprocessed_txt
0,Urban Ladder Eisner Low Back Study-Office Comp...,Household,0,Urban Ladder Eisner Low Study Office Computer ...
1,"Contrast living Wooden Decorative Box,Painted ...",Household,0,contrast live Wooden Decorative Box Painted Bo...
2,IO Crest SY-PCI40010 PCI RAID Host Controller ...,Electronics,2,IO Crest SY PCI40010 PCI RAID Host Controller ...
3,ISAKAA Baby Socks from Just Born to 8 Years- P...,Clothing & Accessories,3,ISAKAA Baby Socks bear 8 Years- Pack 4 6 8 12 ...
4,Indira Designer Women's Art Mysore Silk Saree ...,Clothing & Accessories,3,Indira Designer woman Art Mysore Silk Saree Bl...


In [78]:
X_train, X_test, y_train, y_test = train_test_split(
    df.preprocessed_txt, 
    df.label_num,
    test_size=0.2, # 20% samples will go to test dataset
    random_state=2022,
    stratify=df.label_num
)

In [82]:
from sklearn.ensemble import RandomForestClassifier
clf = Pipeline([
     ('vectorizer_tfidf',TfidfVectorizer()),      
     ('Random Forest', RandomForestClassifier())         
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.96      0.96      1200
           1       0.98      0.98      0.98      1200
           2       0.98      0.98      0.98      1200
           3       0.98      0.99      0.99      1200

    accuracy                           0.98      4800
   macro avg       0.98      0.98      0.98      4800
weighted avg       0.98      0.98      0.98      4800

